<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [5]:
to_drop = ["Latitude", "Longitude", "datetime",
           "is_a_building", "relative_position"
           ]
target = "UHI Index"

df = pd.read_csv("Train_Final.csv").drop(to_drop, axis=1, errors="ignore")

# Uncomment if used
to_drop = np.concatenate((to_drop, df.columns[df.columns.str.contains("count", regex=True)]))
df.drop(columns=to_drop, inplace=True, errors="ignore")

# Experiment using top 80% features
use_cols = [
    target,
    "average_distance",
    "std_distance",
    "distance_range",
    "distance_variation",
    "temp_median",
    "avg_polygon_complexity",
    "nearest_building_size",
    "neighboring_intersection",
    "building_area_density",
]
df = df.loc[:, use_cols]

df.head()

,UHI Index,average_distance,std_distance,distance_range,distance_variation,temp_median,avg_polygon_complexity,nearest_building_size,neighboring_intersection,building_area_density
0,1.030289,3029.494982,3010.415846,6020.831692,3010.415846,38.431539,29.0,14383.618621,14383.618621,1.834324
1,1.030289,3034.387105,3015.153812,6030.307623,3015.153812,38.431539,29.0,14383.618621,14383.618621,1.834324
2,1.023798,3040.018287,3019.750278,6039.500556,3019.750278,37.785534,29.0,14383.618621,14383.618621,1.834324
3,1.023798,3045.783857,3024.815152,6049.630304,3024.815152,37.785534,29.0,14383.618621,14383.618621,2.163022
4,1.021634,3048.490175,3032.165299,6064.330598,3032.165299,37.785534,29.0,14383.618621,14383.618621,2.163022


In [6]:
def create_train(df_features, scaler, train_size=0.8, indices=None):
    print("Removing duplicates...")
    rows_before = df_features.shape[0]
    check_dupl = df_features.columns[1:]
    df_features = df_features.drop_duplicates(subset=check_dupl, keep='first')
    rows_after = df_features.shape[0]
    print(f"Removed {rows_before-rows_after} duplicate rows!")

    X = df_features.drop(target, axis=1)
    y = df_features[target]

    print("Scaling...")
    if indices is not None:
        X_train, X_test, y_train, y_test = X.loc[indices[0]], X.loc[indices[1]], y.loc[indices[0]], y.loc[indices[1]]
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)

    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    print("Done")
    return X_train, X_test, y_train, y_test, scaler
scaler = StandardScaler()
train_index = pd.read_csv("Train_DeterministicIndex.csv").Index
test_index = pd.read_csv("Test_DeterministicIndex.csv").Index
X_train, X_test, y_train, y_test, scaler = create_train(df, scaler, indices=(train_index, test_index))

Removing duplicates...
Removed 0 duplicate rows!
Scaling...
Done


# Modelling

In [8]:
from sklearn.metrics import r2_score


def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

In [9]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from tqdm import tqdm
models = [
    {"model": RandomForestRegressor(200, random_state=0, n_jobs=-1)},
    {"model": RandomForestRegressor(250, random_state=0, n_jobs=-1)},
    {"model": RandomForestRegressor(300, random_state=0, n_jobs=-1)},
    {"model": RandomForestRegressor(150, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1, bootstrap=True, oob_score=True)},
    {"model": ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=250, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=-1)},
    {"model": KNeighborsRegressor(3, n_jobs=-1)},
    {"model": KNeighborsRegressor(5, n_jobs=-1)},
    {"model": DecisionTreeRegressor(random_state=0)},
    # {"model": MLPRegressor((8, 16, 32), random_state=0)},
]

for model in tqdm(models):
    insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
    model["insample"] = insample
    model["outsample"] = outsample

results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
results
# 1.000000	0.934180 | 1.000000	0.942341

100%|██████████| 11/11 [01:35<00:00,  8.65s/it]


,model,insample,outsample
0,"(ExtraTreeRegressor(random_state=209652396), E...",1.000000,0.954515
1,"(ExtraTreeRegressor(random_state=209652396), E...",1.000000,0.953847
2,"(ExtraTreeRegressor(random_state=209652396), E...",1.000000,0.953829
3,"(ExtraTreeRegressor(random_state=209652396), E...",0.992220,0.941746
4,"(DecisionTreeRegressor(max_features=1.0, rando...",0.990933,0.931253
5,"(DecisionTreeRegressor(max_features=1.0, rando...",0.990918,0.930802
6,"(DecisionTreeRegressor(max_features=1.0, rando...",0.990679,0.930630
7,"(DecisionTreeRegressor(max_features=1.0, rando...",0.990767,0.930138
8,DecisionTreeRegressor(random_state=0),1.000000,0.858745
9,"KNeighborsRegressor(n_jobs=-1, n_neighbors=3)",0.948851,0.855806


In [10]:
best_model = results.iloc[0]
best_model.model

ExtraTreesRegressor(n_jobs=-1, random_state=0)

In [11]:
importance = pd.DataFrame(
    {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
).sort_values("Importance", ascending=False).reset_index(drop=True)
importance["cumulative_importance"] = importance.Importance.cumsum() / importance.Importance.sum()
importance

,Features,Importance,cumulative_importance
0,distance_range,0.159954,0.159954
1,average_distance,0.159074,0.319028
2,distance_variation,0.156514,0.475542
3,std_distance,0.145467,0.621009
4,temp_median,0.099735,0.720743
5,nearest_building_size,0.070737,0.791480
6,neighboring_intersection,0.069904,0.861384
7,avg_polygon_complexity,0.069320,0.930704
8,building_area_density,0.069296,1.000000


# Predicting Submission

In [12]:
def create_submission(filename: str, model, scaler):
    sub_df = pd.read_csv("Submission_Final.csv")
    final_df = sub_df[["Latitude", "Longitude"]].copy()
    print("Predicting", sub_df.shape[0], "rows...")

    to_predict = pd.DataFrame(
        scaler.transform(sub_df.loc[:, X_train.columns]),
        columns=X_train.columns
    )

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return
create_submission("BestModel_Experiment80pctgFtrs.csv", best_model.model, scaler)

Predicting 1040 rows...
Predicting...
Done!


---